In [1]:
import os
import json as _json
from pathlib import Path
from dataclasses import dataclass, field
from typing import cast
from typing import List, Optional, Dict, Any, Tuple, Union, Callable, Set
import pandas as pd
import polars as pl
import io
import numpy as np
from types import SimpleNamespace
from polars.testing import assert_frame_equal as pl_assert_frame_equal
import logging
print('pandas:', pd.__version__, ' polars:', pl.__version__)

/opt/anaconda3/envs/my_nlp_env/lib/python3.12/site-packages/pandas/core/computation/expressions.py:22: UserWarning: Pandas requires version '2.10.2' or newer of 'numexpr' (version '2.8.7' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED
/opt/anaconda3/envs/my_nlp_env/lib/python3.12/site-packages/pandas/core/arrays/masked.py:56: UserWarning: Pandas requires version '1.4.2' or newer of 'bottleneck' (version '1.3.7' currently installed).
  from pandas.core import (


pandas: 3.0.2  polars: 1.39.3


In [2]:
# ── Fixtures ────────────────────────────────────────────────────────────────

# --- scoring_compare_methylation_pattern_migration ---
FIX_SCORING_COMPARE_METHYLATION_PATTERN_MIGRATION_CHOICES = [0, 1, 1, 0, 2, 3]
FIX_SCORING_COMPARE_METHYLATION_PATTERN_MIGRATION_MOTIF_BINARY_COMPARE = pd.DataFrame({
    "bin": ["b1", "b1", "b2", "b2", "b3", "b3"],
    "bin_compare": ["c1", "c1", "c2", "c2", "c3", "c3"],
    "methylation_binary": [1, 1, 0, 0, 1, 0],
    "methylation_binary_compare": [1, 0, 1, 0, np.nan, np.nan],
})
FIX_SCORING_COMPARE_METHYLATION_PATTERN_MIGRATION_MOTIF_BINARY_COMPARE_PL = pl.from_pandas(FIX_SCORING_COMPARE_METHYLATION_PATTERN_MIGRATION_MOTIF_BINARY_COMPARE)

# --- scoring_compare_methylation_pattern_multiprocessed_migration ---
class _FakePool:
    def __init__(self, processes=1):
        self.processes = processes
    def __enter__(self):
        return self
    def __exit__(self, exc_type, exc, tb):
        return False
    def starmap(self, func, iterable):
        return [func(*args) for args in iterable]

Pool = _FakePool

def FIX_SCORING_COMPARE_METHYLATION_PATTERN_MULTIPROCESSED_MIGRATION_PROCESS_BIN_CONTIG(bin_contig, bin_consensus, motifs_scored_in_contigs, choices):
    if isinstance(motifs_scored_in_contigs, pl.DataFrame):
        return pl.DataFrame({"bin_compare": [bin_contig], "score": [1]}), None
    return pd.DataFrame({"bin_compare": [bin_contig], "score": [1]}), None

FIX_SCORING_COMPARE_METHYLATION_PATTERN_MULTIPROCESSED_MIGRATION_POOL = Pool

# --- scoring_define_mean_methylation_thresholds_migration ---
FIX_SCORING_DEFINE_MEAN_METHYLATION_THRESHOLDS_MIGRATION_MOTIF_BINARY_COMPARE = pd.DataFrame({
    "mean_methylation": [0.8, 0.2, 0.9, 0.1],
    "methylation_binary": [1, 0, 1, 0],
    "std_methylation_bin": [0.05, 0.1, 0.03, 0.02],
    "mean": [0.75, 0.25, 0.35, 0.10],
})
FIX_SCORING_DEFINE_MEAN_METHYLATION_THRESHOLDS_MIGRATION_MOTIF_BINARY_COMPARE_PL = pl.from_pandas(FIX_SCORING_DEFINE_MEAN_METHYLATION_THRESHOLDS_MIGRATION_MOTIF_BINARY_COMPARE)

# --- scoring_process_bin_contig_merge_migration ---
FIX_SCORING_PROCESS_BIN_CONTIG_MERGE_MIGRATION_BIN_CONTIG = "contig_1"
FIX_SCORING_PROCESS_BIN_CONTIG_MERGE_MIGRATION_BIN_MOTIFS_FROM_MOTIFS_SCORED_IN_BINS = pd.DataFrame({"motif_mod": ["m1", "m2"], "bin": ["b1", "b1"], "methylation_binary": [1, 0]})
FIX_SCORING_PROCESS_BIN_CONTIG_MERGE_MIGRATION_BIN_MOTIFS_FROM_MOTIFS_SCORED_IN_BINS_PL = pl.from_pandas(FIX_SCORING_PROCESS_BIN_CONTIG_MERGE_MIGRATION_BIN_MOTIFS_FROM_MOTIFS_SCORED_IN_BINS)
FIX_SCORING_PROCESS_BIN_CONTIG_MERGE_MIGRATION_MOTIFS_SCORED_IN_CONTIGS = pd.DataFrame({"motif_mod": ["m1", "m2"], "bin_compare": ["contig_1", "contig_1"], "methylation_binary_compare": [1, 0]})
FIX_SCORING_PROCESS_BIN_CONTIG_MERGE_MIGRATION_MOTIFS_SCORED_IN_CONTIGS_PL = pl.from_pandas(FIX_SCORING_PROCESS_BIN_CONTIG_MERGE_MIGRATION_MOTIFS_SCORED_IN_CONTIGS)
FIX_SCORING_PROCESS_BIN_CONTIG_MERGE_MIGRATION_LOG_QUEUE = None
FIX_SCORING_PROCESS_BIN_CONTIG_MERGE_MIGRATION_WORKER_SETUP_LOGGING = lambda log_queue: None

print("✅ Fixtures loaded")


✅ Fixtures loaded


In [3]:
# ── Before wrappers (verbatim pandas) ───────────────────────────────────────

def before_scoring_compare_methylation_pattern_migration(choices, motif_binary_compare):
    # Define conditions and compute comparison scores using numpy.select
    conditions = [
        (motif_binary_compare["methylation_binary"] == 1) & (motif_binary_compare["methylation_binary_compare"] == 1),
        (motif_binary_compare["methylation_binary"] == 1) & (motif_binary_compare["methylation_binary_compare"] == 0),
        (motif_binary_compare["methylation_binary"] == 0) & (motif_binary_compare["methylation_binary_compare"] == 1),
        (motif_binary_compare["methylation_binary"] == 0) & (motif_binary_compare["methylation_binary_compare"] == 0),
        (motif_binary_compare["methylation_binary"] == 1) & (motif_binary_compare["methylation_binary_compare"].isna()),
        (motif_binary_compare["methylation_binary"] == 0) & (motif_binary_compare["methylation_binary_compare"].isna()),
    ]
    motif_binary_compare["motif_comparison_score"] = np.select(conditions, choices, default=np.nan)
    contig_bin_comparison_score = motif_binary_compare.groupby(["bin", "bin_compare"]).agg(
        binary_methylation_missmatch_score=pd.NamedAgg(column="motif_comparison_score", aggfunc="sum"),
        non_na_comparisons=pd.NamedAgg(column="motif_comparison_score", aggfunc="count")
    ).reset_index()
    return contig_bin_comparison_score

def before_scoring_compare_methylation_pattern_multiprocessed_migration(pool, process_bin_contig):
    def compare_methylation_pattern_multiprocessed(motifs_scored_in_bins, bin_consensus, choices, args, num_processes=1):
        logger = logging.getLogger(__name__)
        logger.info("Starting comparison of methylation patterns")

        motifs_scored_in_contigs = motifs_scored_in_bins[motifs_scored_in_bins["n_motifs"] >= args.n_motif_contig_cutoff]
        motifs_scored_in_contigs = motifs_scored_in_contigs[["bin_contig", "motif_mod", "mean"]]
        motifs_scored_in_contigs.rename(columns={"bin_contig": "bin_compare"}, inplace=True)

        comparison_score = pd.DataFrame()
        contigs_w_no_methylation = []

        with Pool(processes=num_processes) as pool:
            results = pool.starmap(
                process_bin_contig,
                [
                    (bin_contig, bin_consensus, motifs_scored_in_contigs, choices)
                    for bin_contig in motifs_scored_in_contigs["bin_compare"].unique()
                ]
            )

        for result, no_methylation in results:
            if result is not None:
                comparison_score = pd.concat([comparison_score, result])
            if no_methylation is not None:
                contigs_w_no_methylation.append(no_methylation)

        return comparison_score, contigs_w_no_methylation
    return compare_methylation_pattern_multiprocessed

def before_scoring_define_mean_methylation_thresholds_migration(motif_binary_compare):
    motif_binary_compare["methylation_mean_threshold"] = np.where(
        motif_binary_compare["methylation_binary"] == 1,
        np.maximum(motif_binary_compare["mean_methylation"] - 4 * motif_binary_compare["std_methylation_bin"], 0.1),
        np.nan
    )
    motif_binary_compare["methylation_binary_compare"] = np.where(
        (motif_binary_compare["methylation_binary"] == 1) & 
        ((motif_binary_compare["mean"] >= motif_binary_compare["methylation_mean_threshold"]) | 
        (motif_binary_compare["mean"] > 0.4)),
        1,
        np.where(motif_binary_compare["methylation_binary"] == 1, 0, np.nan)
    )
    motif_binary_compare["methylation_mean_threshold"] = np.where(
        motif_binary_compare["methylation_binary"] == 0,
        0.25,
        motif_binary_compare["methylation_mean_threshold"]
    )
    motif_binary_compare["methylation_binary_compare"] = np.where(
        motif_binary_compare["methylation_binary"] == 0,
        (motif_binary_compare["mean"] >= 0.25).astype(int),
        motif_binary_compare["methylation_binary_compare"]
    )
    return None

def before_scoring_process_bin_contig_merge_migration(bin_contig, bin_motifs_from_motifs_scored_in_bins, log_queue, motifs_scored_in_contigs, worker_setup_logging):
    worker_setup_logging(log_queue)
    motif_binary_compare = pd.merge(
        bin_motifs_from_motifs_scored_in_bins,
        motifs_scored_in_contigs[motifs_scored_in_contigs["bin_compare"] == bin_contig],
        on="motif_mod"
    )
    contigHasNMethylation = motif_binary_compare["methylation_binary_compare"].sum()
    return contigHasNMethylation

In [4]:
# ── Generated wrappers (verbatim LLM-generated Polars) ──────────────────────

def gen_scoring_compare_methylation_pattern_migration(choices, motif_binary_compare):
    import numpy as np

    motif_binary_compare = motif_binary_compare.with_columns(
        pl.when(
            (pl.col("methylation_binary") == 1) & (pl.col("methylation_binary_compare") == 1)
        )
        .then(choices[0])
        .when(
            (pl.col("methylation_binary") == 1) & (pl.col("methylation_binary_compare") == 0)
        )
        .then(choices[1])
        .when(
            (pl.col("methylation_binary") == 0) & (pl.col("methylation_binary_compare") == 1)
        )
        .then(choices[2])
        .when(
            (pl.col("methylation_binary") == 0) & (pl.col("methylation_binary_compare") == 0)
        )
        .then(choices[3])
        .when(
            (pl.col("methylation_binary") == 1) & (pl.col("methylation_binary_compare").is_null())
        )
        .then(choices[4])
        .when(
            (pl.col("methylation_binary") == 0) & (pl.col("methylation_binary_compare").is_null())
        )
        .then(choices[5])
        .otherwise(np.nan)
        .alias("motif_comparison_score")
    )

    contig_bin_comparison_score = (
        motif_binary_compare.group_by(["bin", "bin_compare"])
        .agg(
            binary_methylation_missmatch_score=pl.col("motif_comparison_score")
            .fill_nan(None)
            .sum(),
            non_na_comparisons=pl.col("motif_comparison_score")
            .fill_nan(None)
            .count(),
        )
        .sort(["bin", "bin_compare"])
    )
    return contig_bin_comparison_score

def gen_scoring_compare_methylation_pattern_multiprocessed_migration(pool, process_bin_contig):
    import logging
    from multiprocessing import Pool



    def compare_methylation_pattern_multiprocessed(motifs_scored_in_bins, bin_consensus, choices, args, num_processes=1):
        logger = logging.getLogger(__name__)
        logger.info("Starting comparison of methylation patterns")

        motifs_scored_in_contigs = motifs_scored_in_bins.filter(pl.col("n_motifs") >= args.n_motif_contig_cutoff)
        motifs_scored_in_contigs = motifs_scored_in_contigs.select(["bin_contig", "motif_mod", "mean"])
        motifs_scored_in_contigs = motifs_scored_in_contigs.rename({"bin_contig": "bin_compare"})

        comparison_score = pl.DataFrame()
        contigs_w_no_methylation = []

        with Pool(processes=num_processes) as pool:
            results = pool.starmap(
                process_bin_contig,
                [
                    (bin_contig, bin_consensus, motifs_scored_in_contigs, choices)
                    for bin_contig in motifs_scored_in_contigs.get_column("bin_compare").unique(maintain_order=True).to_list()
                ],
            )

        for result, no_methylation in results:
            if result is not None:
                comparison_score = pl.concat([comparison_score, result], how="vertical")
            if no_methylation is not None:
                contigs_w_no_methylation.append(no_methylation)

        return comparison_score, contigs_w_no_methylation
    return compare_methylation_pattern_multiprocessed

def gen_scoring_define_mean_methylation_thresholds_migration(motif_binary_compare):

    motif_binary_compare = motif_binary_compare.with_columns(
        pl.when(pl.col("methylation_binary") == 1)
        .then(
            pl.when(
                (pl.col("mean_methylation") - 4 * pl.col("std_methylation_bin")) < 0.1
            )
            .then(0.1)
            .otherwise(pl.col("mean_methylation") - 4 * pl.col("std_methylation_bin"))
        )
        .otherwise(pl.lit(float("nan")))
        .alias("methylation_mean_threshold")
    ).with_columns(
        pl.when(
            (pl.col("methylation_binary") == 1)
            & (
                (pl.col("mean") >= pl.col("methylation_mean_threshold"))
                | (pl.col("mean") > 0.4)
            )
        )
        .then(1)
        .otherwise(
            pl.when(pl.col("methylation_binary") == 1)
            .then(0)
            .otherwise(pl.lit(float("nan")))
        )
        .alias("methylation_binary_compare")
    ).with_columns(
        pl.when(pl.col("methylation_binary") == 0)
        .then(0.25)
        .otherwise(pl.col("methylation_mean_threshold"))
        .alias("methylation_mean_threshold")
    ).with_columns(
        pl.when(pl.col("methylation_binary") == 0)
        .then((pl.col("mean") >= 0.25).cast(pl.Int64))
        .otherwise(pl.col("methylation_binary_compare"))
        .alias("methylation_binary_compare")
    )
    return motif_binary_compare

def gen_scoring_process_bin_contig_merge_migration(bin_contig, bin_motifs_from_motifs_scored_in_bins, log_queue, motifs_scored_in_contigs, worker_setup_logging):
    worker_setup_logging(log_queue)
    motif_binary_compare = bin_motifs_from_motifs_scored_in_bins.join(
        motifs_scored_in_contigs.filter(pl.col("bin_compare") == bin_contig),
        on="motif_mod",
        how="inner",
    )
    contigHasNMethylation = motif_binary_compare.select(
        pl.col("methylation_binary_compare").sum()
    ).item()
    return contigHasNMethylation

In [5]:
# ── Comparison helper ───────────────────────────────────────────────────────
def _index_is_trivial(idx):
    # Unnamed + integer-valued covers both a fresh RangeIndex and the leftover
    # positional index after filtering/boolean-masking a RangeIndex-based frame
    # (pandas downgrades RangeIndex to a plain Int64Index on filter, but it's
    # still just leftover row positions, not real data). A set_index(...)
    # always carries the original column's name, so any genuinely meaningful
    # index is caught by the "name is not None" branch.
    return idx.name is None and pd.api.types.is_integer_dtype(idx.dtype)


def _to_pl(r):
    if isinstance(r, pl.DataFrame): return r
    if isinstance(r, pd.DataFrame): return pl.from_pandas(r.reset_index(drop=True) if _index_is_trivial(r.index) else r.reset_index())
    if isinstance(r, pd.Series): return pl.from_pandas(r.to_frame().reset_index(drop=True) if _index_is_trivial(r.index) else r.to_frame().reset_index())
    return None

def compare(before_result, gen_result, label, check_row_order=False):
    raw_label = str(label)
    label_parts = raw_label.strip().split()
    is_l3 = bool(label_parts and label_parts[0].upper() == "L3")
    layer = "L3" if is_l3 else "L2"
    kind = "edge" if is_l3 else "equivalence"
    if is_l3:
        label_parts = label_parts[1:]
        if label_parts and label_parts[0].lower() in ("edge", "branch"):
            label_parts = label_parts[1:]
        display_label = " ".join(label_parts)
    else:
        display_label = raw_label

    left  = _to_pl(before_result.collect() if isinstance(before_result, pl.LazyFrame) else before_result)
    right = _to_pl(gen_result.collect() if isinstance(gen_result, pl.LazyFrame) else gen_result)
    if left is None and right is None:
        print(f"⚠️  {layer} {kind} {display_label}: both sides non-DataFrame (no output to compare)")
        return
    if left is None or right is None:
        print(f"❌ {layer} {kind} {display_label}: MISMATCH — one side returned DataFrame, other did not")
        return
    left_cols, right_cols = set(left.columns), set(right.columns)
    if left_cols != right_cols:
        print(f"❌ {layer} {kind} {display_label}: MISMATCH — column sets differ (before-only={left_cols - right_cols}, gen-only={right_cols - left_cols})")
        return
    common = list(left.columns)
    try:
        pl_assert_frame_equal(left.select(common), right.select(common),
                              check_dtypes=False, check_row_order=check_row_order)
        print(f"✅ {layer} {kind} {display_label}: MATCH")
    except Exception as e:
        print(f"❌ {layer} {kind} {display_label}: MISMATCH — {e}")


In [6]:
# === Tests: scoring_define_mean_methylation_thresholds_migration ===

# L1 smoke – generated
try:
    _r = gen_scoring_define_mean_methylation_thresholds_migration(FIX_SCORING_DEFINE_MEAN_METHYLATION_THRESHOLDS_MIGRATION_MOTIF_BINARY_COMPARE_PL)
    print("✅ L1 smoke gen_scoring_define_mean_methylation_thresholds_migration: OK, type=", type(_r).__name__)
except Exception as _e:
    print(f"❌ L1 smoke gen_scoring_define_mean_methylation_thresholds_migration: {type(_e).__name__}: {_e}")

# L1 smoke – before
try:
    _rb = before_scoring_define_mean_methylation_thresholds_migration(FIX_SCORING_DEFINE_MEAN_METHYLATION_THRESHOLDS_MIGRATION_MOTIF_BINARY_COMPARE)
    print("✅ L1 smoke before_scoring_define_mean_methylation_thresholds_migration: OK")
except Exception as _e:
    print(f"❌ L1 smoke before_scoring_define_mean_methylation_thresholds_migration: {type(_e).__name__}: {_e}")

# L2 behavioral equivalence - compare the mutated pandas state with
# the replacement Polars frame returned by the immutable implementation.
try:
    _before_input = FIX_SCORING_DEFINE_MEAN_METHYLATION_THRESHOLDS_MIGRATION_MOTIF_BINARY_COMPARE.copy()
    before_scoring_define_mean_methylation_thresholds_migration(_before_input)
    _rg = gen_scoring_define_mean_methylation_thresholds_migration(
        FIX_SCORING_DEFINE_MEAN_METHYLATION_THRESHOLDS_MIGRATION_MOTIF_BINARY_COMPARE_PL
    )
    compare(_before_input, _rg, "scoring_define_mean_methylation_thresholds_migration")
except Exception as _e:
    print(f"❌ L2 equivalence scoring_define_mean_methylation_thresholds_migration: setup error — {type(_e).__name__}: {_e}")


# L3 branch – pandas mutating function versus generated returned Polars DataFrame
try:
    _pd_input = FIX_SCORING_DEFINE_MEAN_METHYLATION_THRESHOLDS_MIGRATION_MOTIF_BINARY_COMPARE.copy()
    before_scoring_define_mean_methylation_thresholds_migration(_pd_input)
    _gen_edge = gen_scoring_define_mean_methylation_thresholds_migration(
        FIX_SCORING_DEFINE_MEAN_METHYLATION_THRESHOLDS_MIGRATION_MOTIF_BINARY_COMPARE_PL
    )
    compare(_pd_input, _gen_edge, "L3 branch scoring_define_mean_methylation_thresholds_migration")
except Exception as _e:
    print(f"❌ L3 branch scoring_define_mean_methylation_thresholds_migration: {type(_e).__name__}: {_e}")


✅ L1 smoke gen_scoring_define_mean_methylation_thresholds_migration: OK, type= DataFrame
✅ L1 smoke before_scoring_define_mean_methylation_thresholds_migration: OK
✅ L2 equivalence scoring_define_mean_methylation_thresholds_migration: MATCH
✅ L3 edge scoring_define_mean_methylation_thresholds_migration: MATCH
